In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q kagglehub scikit-learn matplotlib seaborn tqdm

In [ ]:
import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("All libraries imported successfully!")

In [ ]:
# CELL 4: Download dataset

from google.colab import userdata
import os

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

import kagglehub

path = kagglehub.dataset_download(
    "andrewmvd/lung-and-colon-cancer-histopathological-images"
)

print("Dataset downloaded to:", path)

In [ ]:
import os

dataset_path = path

ACA_PATH = None
BENIGN_PATH = None

for root, dirs, files in os.walk(dataset_path):
    if "colon_aca" in dirs and "colon_n" in dirs:
        ACA_PATH = os.path.join(root, "colon_aca")
        BENIGN_PATH = os.path.join(root, "colon_n")
        break

In [ ]:
# CELL 6: Project configuration

IMAGE_SIZE = 224
BATCH_SIZE = 32

EPOCHS = 25
FINE_TUNE_EPOCHS = 10

LEARNING_RATE = 0.0001
FINE_TUNE_LR = 0.00001

CLASS_NAMES = [
    "Adenocarcinoma",
    "Benign"
]

CLASS_TO_IDX = {
    "Adenocarcinoma": 0,
    "Benign": 1
}

OUTPUT_DIR = "/content/colorectal_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    OUTPUT_DIR, "best_colorectal_model.pt"
)

FINAL_MODEL_PATH = os.path.join(
    OUTPUT_DIR, "colorectal_cancer_efficientnet.pt"
)

print("Configuration completed!")
print("Output directory:", OUTPUT_DIR)

In [ ]:
# CELL 7: Collect images

image_paths = []
labels = []

def collect_images(folder, label, max_images=5000):

    valid_extensions = {".jpg", ".jpeg", ".png"}

    files = sorted([
        f for f in Path(folder).iterdir()
        if f.is_file() and f.suffix.lower() in valid_extensions
    ])

    if len(files) < max_images:
        raise ValueError(
            f"Expected {max_images} images in {folder}, found {len(files)}"
        )

    # Reproducible selection
    rng = random.Random(SEED + label)
    selected_files = rng.sample(files, max_images)

    for file in selected_files:
        image_paths.append(str(file))
        labels.append(label)

collect_images(ACA_PATH, 0)
collect_images(BENIGN_PATH, 1)

image_paths = np.array(image_paths)
labels = np.array(labels)

print("Total selected images:", len(image_paths))
print("Adenocarcinoma:", np.sum(labels == 0))
print("Benign:", np.sum(labels == 1))

In [ ]:
# CELL 8: Class distribution

counts = [
    np.sum(labels == 0),
    np.sum(labels == 1)
]

plt.figure(figsize=(7, 5))

plt.bar(CLASS_NAMES, counts)

plt.title("Colorectal Dataset Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")

for i, count in enumerate(counts):
    plt.text(i, count + 50, str(count), ha="center")

plt.show()

In [ ]:
# CELL 9: Show sample images

fig, axes = plt.subplots(2, 5, figsize=(15, 7))

for row, class_id in enumerate([0, 1]):

    indices = np.where(labels == class_id)[0]
    selected = np.random.default_rng(SEED).choice(
        indices, 5, replace=False
    )

    for col, idx in enumerate(selected):

        image = Image.open(image_paths[idx]).convert("RGB")

        axes[row, col].imshow(image)
        axes[row, col].set_title(CLASS_NAMES[class_id])
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# CELL 10: Train / Validation / Test split

X_train, X_temp, y_train, y_temp = train_test_split(
    image_paths,
    labels,
    test_size=0.30,
    stratify=labels,
    random_state=SEED
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=SEED
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

print("\nTraining class distribution:")
print(pd.Series(y_train).value_counts().sort_index())

In [ ]:
# CELL 11: Image preprocessing

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])

print("Image transformations ready!")

In [ ]:
# CELL 12: Custom PyTorch Dataset

class ColonCancerDataset(Dataset):

    def __init__(self, paths, labels, transform=None):
        self.paths = list(paths)
        self.labels = list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):

        image = Image.open(
            self.paths[index]
        ).convert("RGB")

        label = int(self.labels[index])

        if self.transform:
            image = self.transform(image)

        return image, label


train_dataset = ColonCancerDataset(
    X_train, y_train, train_transform
)

val_dataset = ColonCancerDataset(
    X_val, y_val, eval_transform
)

test_dataset = ColonCancerDataset(
    X_test, y_test, eval_transform
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

In [ ]:
# CELL 13: DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created successfully!")

In [ ]:
# CELL 14: Load EfficientNetB0

import torch
import torch.nn as nn
from torchvision import models

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

weights = models.EfficientNet_B0_Weights.DEFAULT

model = models.efficientnet_b0(weights=weights)

# Freeze feature extraction layers
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier for two classes
in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    in_features,
    len(CLASS_NAMES)
)

model = model.to(DEVICE)

print("EfficientNetB0 loaded!")
print("Number of output classes:", len(CLASS_NAMES))

In [ ]:
# CELL 15: Training configuration

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.classifier.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

scaler = torch.amp.GradScaler("cuda")

print("Loss function:", criterion)
print("Optimizer:", "AdamW")
print("Mixed precision: Enabled")

In [ ]:
# CELL 16: Training function

def train_one_epoch(model, loader, optimizer, scaler):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    progress = tqdm(loader, desc="Training")

    for images, labels in progress:

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            accuracy=f"{100 * correct / total:.2f}%"
        )

    return total_loss / total, correct / total

In [ ]:
# CELL 17: Validation function

@torch.no_grad()
def validate(model, loader):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Validation"):

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

In [ ]:
# CELL 18: Initial training

best_val_accuracy = -1.0

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print("=" * 55)

    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, scaler
    )

    val_loss, val_acc = validate(
        model, val_loader
    )

    scheduler.step(val_acc)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    print(f"Training Loss: {train_loss:.4f}")
    print(f"Training Accuracy: {train_acc*100:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc*100:.2f}%")

    if val_acc > best_val_accuracy:

        best_val_accuracy = val_acc

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print("Best model checkpoint saved!")

print("\nInitial training completed!")
print("Best validation accuracy:", best_val_accuracy * 100)

In [ ]:
# CELL 19: Fine-tuning setup

model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE,
        weights_only=True
    )
)

# Unfreeze the last two feature blocks
for param in model.features[-2:].parameters():
    param.requires_grad = True

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FINE_TUNE_LR,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

print("Fine-tuning ready!")
print("Trainable feature blocks: Last two")

In [ ]:
# CELL 20: Fine-tuning

for epoch in range(FINE_TUNE_EPOCHS):

    print(f"\nFine-tuning Epoch {epoch + 1}/{FINE_TUNE_EPOCHS}")
    print("=" * 55)

    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, scaler
    )

    val_loss, val_acc = validate(
        model, val_loader
    )

    scheduler.step(val_acc)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    print(f"Training Accuracy: {train_acc*100:.2f}%")
    print(f"Validation Accuracy: {val_acc*100:.2f}%")

    if val_acc > best_val_accuracy:

        best_val_accuracy = val_acc

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print("Improved checkpoint saved!")

print("\nFine-tuning completed!")

In [ ]:
# CELL 21: Restore best checkpoint

model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE,
        weights_only=True
    )
)

model.eval()

print("Best model restored successfully!")
print("Best validation accuracy:", f"{best_val_accuracy*100:.2f}%")

In [ ]:
# CELL 22: Final test evaluation

all_predictions = []
all_labels = []
all_probabilities = []

model.eval()

with torch.no_grad():

    for images, labels in tqdm(test_loader, desc="Testing"):

        images = images.to(DEVICE)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(images)

        probabilities = F.softmax(outputs.float(), dim=1)

        predictions = probabilities.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probabilities.extend(probabilities.cpu().numpy())

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)
all_probabilities = np.array(all_probabilities)

accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(
    all_labels, all_predictions, zero_division=0
)
recall = recall_score(
    all_labels, all_predictions, zero_division=0
)
f1 = f1_score(
    all_labels, all_predictions, zero_division=0
)
auc = roc_auc_score(
    all_labels, all_probabilities[:, 1]
)

print("\nFINAL TEST RESULTS")
print("=" * 45)

print(f"Accuracy  : {accuracy*100:.2f}%")
print(f"Precision : {precision*100:.2f}%")
print(f"Recall    : {recall*100:.2f}%")
print(f"F1 Score  : {f1*100:.2f}%")
print(f"ROC-AUC   : {auc:.4f}")

print("\nDetailed Classification Report:")

print(classification_report(
    all_labels,
    all_predictions,
    target_names=CLASS_NAMES,
    zero_division=0
))

In [ ]:
# CELL 23: Confusion matrix

cm = confusion_matrix(
    all_labels,
    all_predictions
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)

plt.title("Colorectal Cancer Classification")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.tight_layout()
plt.show()

In [ ]:
# CELL 24: Training graphs

epochs_range = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)

plt.plot(
    epochs_range,
    np.array(history["train_accuracy"]) * 100,
    label="Training Accuracy"
)

plt.plot(
    epochs_range,
    np.array(history["val_accuracy"]) * 100,
    label="Validation Accuracy"
)

plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)

plt.plot(
    epochs_range,
    history["train_loss"],
    label="Training Loss"
)

plt.plot(
    epochs_range,
    history["val_loss"],
    label="Validation Loss"
)

plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# CELL 25: Save final model checkpoint

model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE,
        weights_only=True
    )
)

checkpoint = {
    "model_name": "EfficientNetB0",
    "model_state_dict": model.state_dict(),
    "class_names": CLASS_NAMES,
    "image_size": IMAGE_SIZE,
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
    "test_accuracy": float(accuracy),
    "seed": SEED
}

torch.save(checkpoint, FINAL_MODEL_PATH)

print("Final model saved successfully!")
print("File:", FINAL_MODEL_PATH)

print(
    "File size:",
    round(os.path.getsize(FINAL_MODEL_PATH) / (1024 * 1024), 2),
    "MB"
)

In [ ]:
# CELL 26: Save metrics and split information

metrics = pd.DataFrame([{
    "Model": "EfficientNetB0",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1_Score": f1,
    "ROC_AUC": auc,
    "Training_Images": len(X_train),
    "Validation_Images": len(X_val),
    "Testing_Images": len(X_test)
}])

metrics.to_csv(
    os.path.join(OUTPUT_DIR, "evaluation_metrics.csv"),
    index=False
)

split_df = pd.DataFrame({
    "image_path": list(X_train) + list(X_val) + list(X_test),
    "label": list(y_train) + list(y_val) + list(y_test),
    "split": (
        ["train"] * len(X_train)
        + ["validation"] * len(X_val)
        + ["test"] * len(X_test)
    )
})

split_df.to_csv(
    os.path.join(OUTPUT_DIR, "dataset_splits.csv"),
    index=False
)

print(metrics)
print("\nResults saved!")

In [ ]:
# CELL 27: Download final model

from google.colab import files

files.download(FINAL_MODEL_PATH)

In [ ]:
# CELL 28: Upload an image and predict

from google.colab import files

uploaded = files.upload()

image_name = next(iter(uploaded))

image = Image.open(image_name).convert("RGB")

display(image.resize((350, 350)))

input_tensor = eval_transform(image).unsqueeze(0).to(DEVICE)

model.eval()

with torch.no_grad():

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):
        output = model(input_tensor)

    probabilities = F.softmax(output.float(), dim=1)[0]

predicted_index = probabilities.argmax().item()

print("\nPREDICTION RESULT")
print("=" * 40)

print("Predicted class:", CLASS_NAMES[predicted_index])

for index, name in enumerate(CLASS_NAMES):
    print(
        f"{name}: {probabilities[index].item()*100:.2f}%"
    )

print("\nResearch prototype only – not a clinical diagnosis.")